<a href="https://colab.research.google.com/github/pksheaad/Transformers/blob/main/05_finishing_implementation_PK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# import Libraries
import requests
from typing import Dict,List, Any

import torch
import torch.nn as nn
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import RandomSampler


In [2]:
# set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
# import the text data
url = "https://raw.githubusercontent.com/pksheaad/Transformers/refs/heads/main/Data/tiny-shakespeare.txt"

request = requests.get(url = url)
text = request.text
print(text[:50])

First Citizen:
Before we proceed any further, hear


In [5]:
# create encoder and decoder
class CharTokenizer:
  """
  This class will create tokenid encoder, decoder.
  This will laso implement the length of vocabulary
  """
  def __init__(self, vocabulary:List[str]) -> None:
    self.token_id_for_char = {char : token_id for token_id , char in enumerate(vocabulary)}
    self.char_for_token_id = {token_id : char for token_id , char in enumerate(vocabulary)}

  @staticmethod
  def text_to_train(text):
    vocabulary = set(text)
    return CharTokenizer(sorted(list(vocabulary)))

  def enocoder(self, text)->torch.tensor:
    token_ids = []
    for char in text:
      token_ids.append(self.token_id_for_char[char])

    return torch.tensor(token_ids, dtype = torch.long)

  def decoder(self, token_ids:torch.tensor):
    decoded_text = []
    for token in token_ids.tolist():
      decoded_text.append(self.char_for_token_id[token])
    return "".join(decoded_text)

  def get_vocabulary_length(self):
    return len(self.token_id_for_char)


In [7]:
# test the class
tokenizer = CharTokenizer.text_to_train(text)
print(f"Enoded Token Id:{tokenizer.enocoder(text[:50])}")
print(f"Decoded Text: {tokenizer.decoder(tokenizer.enocoder(text[:50]))}")
print(f"Vocabulary Size: {tokenizer.get_vocabulary_length()}")

Enoded Token Id:tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])
Decoded Text: First Citizen:
Before we proceed any further, hear
Vocabulary Size: 65


In [8]:
# Create dataset class
class TokenIdsDataset(Dataset):
  def __init__(self, data, block_size) -> None:
    super().__init__()
    self.data = data
    self.block_size = block_size

  # implement the len method
  def __len__(self):
    return len(self.data) - self.block_size

  # implement the getitem method
  def __getitem__(self, index):
    assert index < len(self.data) - self.block_size
    x = self.data[index : index + self.block_size]
    y = self.data[index + 1 : index + 1 + self.block_size]
    return x, y

In [12]:
# Testing
data = tokenizer.enocoder(text)
dataset = TokenIdsDataset(data = data, block_size = 64)
sampler = RandomSampler(data_source = dataset, replacement = True)
dataloader = DataLoader(dataset = dataset, batch_size = 64, sampler = sampler)
x, y = next(iter(dataloader))
print(f"x[0]:\n {x[0]}")
print(f"y[0]:\n {y[0]}")

x[0]:
 tensor([51, 39, 52, 63,  1, 46, 39, 60, 43,  1, 39, 52, 42,  1, 53, 58, 46, 43,
        56, 57,  1, 51, 59, 57, 58,  1, 57, 47, 58,  1, 58, 46, 43, 56, 43, 11,
         0, 13, 52, 42,  1, 47, 52,  1, 58, 46, 47, 57,  1, 58, 46, 53, 59, 45,
        46, 58,  1, 58, 46, 43, 63,  1, 44, 47])
y[0]:
 tensor([39, 52, 63,  1, 46, 39, 60, 43,  1, 39, 52, 42,  1, 53, 58, 46, 43, 56,
        57,  1, 51, 59, 57, 58,  1, 57, 47, 58,  1, 58, 46, 43, 56, 43, 11,  0,
        13, 52, 42,  1, 47, 52,  1, 58, 46, 47, 57,  1, 58, 46, 53, 59, 45, 46,
        58,  1, 58, 46, 43, 63,  1, 44, 47, 52])


In [13]:
# Model Configuration
# Vocabulary Size : vocab_size : Number of unique token iDS supported by tokenizer
# Context Size: The maximum number of tokens model can see at the time
# Embedding Dimension : Size of embedding vector
# Number of Heads: Number of attention head each process the input independently
# Number of Layers: Number of transforming blocks
# Dropout Rate : proprtion of output layer set to zero
# Use Bias: A boolean indicating whether the linear transformation use bias or not
config = {
    "vocabulary_size" : tokenizer.get_vocabulary_length(),
    "context_size" : 256,
    "embedding_dim" : 768,
    "num_head" : 12,
    "num_layers" : 10,
    "dropout_rate" : 0.1,
    "use_bias" : False
}

config["head_size"] = config["embedding_dim"] // config["num_head"] # 768 // 12

In [22]:
# implement the class AttentionHead
class AttentionHead(nn.Module):
  def __init__(self, config:Dict[str,Any]) -> None:
    super().__init__()

    # define Q, K, and W matrices
    self.Q_weight = nn.Linear(in_features = config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])
    self.K_weight = nn.Linear(in_features = config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])
    self.V_weight = nn.Linear(in_features = config['embedding_dim'], out_features = config['head_size'], bias = config['use_bias'])

    # Create the casual_attention_mask
    casual_attention_mask = torch.tril(torch.ones(config['context_size'], config['context_size']))
    self.register_buffer("casual_attention_mask", casual_attention_mask)

    # dropout layer
    self.dropout = nn.Dropout(p = config['dropout_rate'])

  # Implement the forward method
  def forward(self, input):
    batch_size, token_num, embedding_dim = input.shape
    # Q Matrix
    Q = self.Q_weight(input)
    # K Matrix
    K = self.K_weight(input)
    # V Matric
    V = self.V_weight(input)

    attention_score = Q @ K.transpose(1,2)
    attention_score = attention_score.masked_fill(self.casual_attention_mask[:token_num,:token_num]== 0, - torch.inf)
    attention_score = attention_score / (K.shape[-1] **0.5)
    attention_score = torch.softmax(attention_score, dim = -1)
    attention_score = attention_score @ V
    attention_score = self.dropout(attention_score)
    return attention_score



In [23]:
# Testing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
ah = AttentionHead(config = config)
output = ah(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 64])


In [26]:
# Multi AttentionHead Implementation
class MultiAttentionHead(nn.Module):
  def __init__(self, config) -> None:
    super().__init__()
    attention_heads = [AttentionHead(config=config) for _ in range(config['num_head'])]
    self.heads = nn.ModuleList(attention_heads)
    self.linear_layer = nn.Linear(in_features = config['num_head'] * config['head_size'], out_features = config['embedding_dim'], bias = config['use_bias'])
    self.dropout = nn.Dropout(p = config['dropout_rate'])

  # Implement forward method
  def forward(self, input):
    heads = [head(input) for head in self.heads]
    score_change = torch.cat(heads, dim = -1)
    score_chnage = self.linear_layer(score_change)
    score_change = self.dropout(score_change)

    return score_change


In [28]:
# Testing
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
mah = MultiAttentionHead(config = config)
output = mah(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 768])


In [30]:
from torch.nn.modules.dropout import Dropout
# Implementation FeedForward Class
class FeedForward(nn.Module):
  def __init__(self, config) -> None:
    super().__init__()
    self.linear_layer = nn.Sequential(
        nn.Linear(in_features = config['embedding_dim'], out_features = config['embedding_dim']*4, bias = config['use_bias']),
        nn.GELU(),
        nn.Linear(in_features = config['embedding_dim']*4, out_features = config['embedding_dim'], bias = config['use_bias']),
        nn.Dropout(p = config['dropout_rate']))

  def forward(self, input):
    return self.linear_layer(input)




In [31]:
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
ff = FeedForward(config = config)
output = ff(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 768])


In [34]:
class Block(nn.Module):
  def __init__(self, config = config) -> None:
    super().__init__()
    self.linear_norm_1 = nn.LayerNorm(normalized_shape = config['embedding_dim'])
    self.mha = MultiAttentionHead(config = config)
    self.linear_norm_2 = nn.LayerNorm(normalized_shape = config['embedding_dim'])
    self.ff = FeedForward(config = config)

  def forward(self, input):
    residual = input
    x = self.linear_norm_1(input)
    x = self.mha(x)
    x = residual + x

    residual = x
    x = self.linear_norm_2(x)
    x = self.ff(x)
    x = residual + x
    return x


In [35]:
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
block = Block(config = config)
output = ff(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 768])


In [43]:
class DemoGPT(nn.Module):
  def __init__(self, config) -> None:
    super().__init__()

    # Token Embedding
    self.token_embedding = nn.Embedding(num_embeddings = config['vocabulary_size'], embedding_dim = config['embedding_dim'])

    # Positional Encoding
    self.positional_embedding = nn.Embedding(num_embeddings = config['context_size'], embedding_dim = config['embedding_dim'])

    # Transformer Blocks
    blocks = [Block(config = config) for _ in range(config['num_layers'])]
    self.blocks = nn.Sequential(*blocks)

    # Final Layer Normalization and Linear Layer
    self.norm_layer = nn.LayerNorm(normalized_shape = config['embedding_dim'])
    self.unembedding = nn.Linear(in_features = config['embedding_dim'], out_features = config['vocabulary_size'], bias = config['use_bias'])

  # Implement forward method
  def forward(self, token_ids):
    batch_size, token_num = token_ids.shape
    x = self.token_embedding(token_ids)
    sequence = torch.arange(token_num, device=device)
    x = x + self.positional_embedding(sequence)
    x = self.blocks(x)
    x = self.norm_layer(x)
    x = self.unembedding(x)
    return x


In [44]:
# Testing
gpt = DemoGPT(config = config).to(device)
output = gpt(tokenizer.enocoder("Hii Prashant").unsqueeze(dim = 0).to(device))
print(output.shape)


torch.Size([1, 12, 65])
